# Verify TimeSeriesDataset

## Imports

In [1]:
import sys
sys.path.insert(0, '/Users/shelleygoel/Code/01_statistical_mod_blog/TSB-AD/explorations')
sys.path.insert(0, '/Users/shelleygoel/Code/01_statistical_mod_blog/anomaly_detection')

import hvac_data_gen as hvdg
from core.dataset import TimeSeriesDataset
from core.models import EuclideanDistModel
from core.evaluation import Evaluation
from datetime import datetime
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Generate test data with lag anomaly
gen = hvdg.HVACDataGenerator(seed=42)

lag_config = [{
    'unit': 1, 'type': 'lag',
    'start_day': 2, 'start_hour': 8,
    'duration_hours': 48,
    'params': {'lag_minutes': 180}
}]
df_lag = gen.generate_container_data(
    container_id=0, start_time=datetime(2026, 1, 15),
    duration_days=5, anomaly_config=lag_config
)

# Normal container
df_normal = gen.generate_container_data(
    container_id=1, start_time=datetime(2026, 1, 15),
    duration_days=5, anomaly_config=[]
)

hvac_df = pd.concat([df_lag, df_normal], ignore_index=True)
print(f"Shape: {hvac_df.shape}")
print(f"Columns: {list(hvac_df.columns)}")
hvac_df.head()

Shape: (43200, 6)
Columns: ['timestamp_et', 'HVACNum', 'TmpRet', 'anomaly', 'anomaly_type', 'container_id']


,timestamp_et,HVACNum,TmpRet,anomaly,anomaly_type,container_id
0,2026-01-15 00:00:00,0,49.585463,False,normal,0
1,2026-01-15 00:00:00,1,50.828930,False,normal,0
2,2026-01-15 00:00:00,2,50.299590,False,normal,0
3,2026-01-15 00:01:00,0,49.531567,False,normal,0
4,2026-01-15 00:01:00,1,50.915807,False,normal,0


In [3]:
# Create dataset
# Adapt col names to what hvac_data_gen actually produces
print("Unique anomaly_type values:", hvac_df['anomaly_type'].unique())
print("Unique unit values:", hvac_df['unit'].unique() if 'unit' in hvac_df.columns else 'NO unit col')
print("Unique HVACNum values:", hvac_df['HVACNum'].unique() if 'HVACNum' in hvac_df.columns else 'NO HVACNum col')

Unique anomaly_type values: <ArrowStringArray>
['normal', 'lag']
Length: 2, dtype: str
Unique unit values: NO unit col
Unique HVACNum values: [0 1 2]


In [4]:
hvac_df['humidity'] = np.random.uniform(30, 40, len(hvac_df))

In [5]:
hvac_df.head()

,timestamp_et,HVACNum,TmpRet,anomaly,anomaly_type,container_id,humidity
0,2026-01-15 00:00:00,0,49.585463,False,normal,0,33.943878
1,2026-01-15 00:00:00,1,50.828930,False,normal,0,33.674344
2,2026-01-15 00:00:00,2,50.299590,False,normal,0,37.032612
3,2026-01-15 00:01:00,0,49.531567,False,normal,0,39.902232
4,2026-01-15 00:01:00,1,50.915807,False,normal,0,34.053538


# TODO:
- Test plotting of multivariate Timeseries, color coded by subentity
- Create an EDA class - which has some standard figures for the dataset.

In [6]:
# Determine correct column names from data
sub_entity_col = 'unit' if 'unit' in hvac_df.columns else 'HVACNum'
label_col = 'anomaly' if 'anomaly' in hvac_df.columns else 'label'

col_map = {
    'entity': 'container_id',
    'time': 'timestamp_et',
    'value_cols': ['TmpRet', 'humidity'],
    'label': label_col,
    'label_type': 'anomaly_type',
    'sub_entity': sub_entity_col,
}

ds = TimeSeriesDataset(hvac_df, col_map)
print("Created successfully!")

Created successfully!


In [7]:
# Test entities()

# Test anomaly_types()
print("Anomaly types:", ds.anomaly_types())
assert 'normal' in ds.anomaly_types()
assert 'lag' in ds.anomaly_types()

Anomaly types: <ArrowStringArray>
['normal', 'lag']
Length: 2, dtype: str


In [8]:
ds.anomaly_summary()

,label_type,entity_count
0,lag,1
1,normal,1


In [9]:
# Test day_labels()
dl = ds.day_labels()
print("day_labels columns:", list(dl.columns))
print(dl)

# Container 0 should have anomaly on days 2-3 (lag starts day 2 hour 8, 48h)
# Container 1 should have no anomalies
c0_labels = dl[dl['container_id'] == 0]
c1_labels = dl[dl['container_id'] == 1]
print("\nContainer 0 anomaly days:", c0_labels[c0_labels['anomaly'] > 0]['day'].tolist())
print("Container 1 anomaly days:", c1_labels[c1_labels['anomaly'] > 0]['day'].tolist())

day_labels columns: ['container_id', 'day', 'anomaly', 'anomaly_type']
   container_id         day  anomaly anomaly_type
0             0  2026-01-15    False       normal
1             0  2026-01-16    False       normal
2             0  2026-01-17     True          lag
3             0  2026-01-18     True          lag
4             0  2026-01-19     True          lag
5             1  2026-01-15    False       normal
6             1  2026-01-16    False       normal
7             1  2026-01-17    False       normal
8             1  2026-01-18    False       normal
9             1  2026-01-19    False       normal

Container 0 anomaly days: [datetime.date(2026, 1, 17), datetime.date(2026, 1, 18), datetime.date(2026, 1, 19)]
Container 1 anomaly days: []


In [10]:
# Test ts_labels()
tl = ds.ts_labels()
print("ts_labels columns:", list(tl.columns))
print("ts_labels shape:", tl.shape)
print("\nAnomaly timestamps (first 5):")
print(tl[tl['anomaly'] > 0].head())

ts_labels columns: ['container_id', 'timestamp_et', 'anomaly', 'anomaly_type']
ts_labels shape: (14400, 4)

Anomaly timestamps (first 5):
      container_id        timestamp_et  anomaly anomaly_type
3360             0 2026-01-17 08:00:00     True          lag
3361             0 2026-01-17 08:01:00     True          lag
3362             0 2026-01-17 08:02:00     True          lag
3363             0 2026-01-17 08:03:00     True          lag
3364             0 2026-01-17 08:04:00     True          lag


In [11]:
# Test sample_and_visualize_cases
figs = ds.sample_and_visualize_cases(n_cases=2, label_type='lag')
for fig in figs:
    fig.show()

In [12]:
# Validation: check col_map validation catches errors
try:
    TimeSeriesDataset(hvac_df, {'entity': 'container_id'})  # missing required keys
except ValueError as e:
    print(f"Caught expected error: {e}")

try:
    bad_map = {**col_map, 'bogus': 'foo'}
    TimeSeriesDataset(hvac_df, bad_map)  # unknown key
except ValueError as e:
    print(f"Caught expected error: {e}")

try:
    bad_map = {**col_map, 'entity': 'nonexistent_col'}
    TimeSeriesDataset(hvac_df, bad_map)  # missing column
except ValueError as e:
    print(f"Caught expected error: {e}")

Caught expected error: col_map missing required keys: {'value_cols', 'time'}
Caught expected error: col_map has unknown keys: {'bogus'}
Caught expected error: Columns not found in DataFrame: ['nonexistent_col']


# Model Class Test
- test model class works with dataset class
- and outputs a score
- 

In [13]:

sub_entity_col = 'unit' if 'unit' in hvac_df.columns else 'HVACNum'
label_col = 'anomaly' if 'anomaly' in hvac_df.columns else 'label'

col_map = {
    'entity': 'container_id',
    'time': 'timestamp_et',
    'value_cols': ['TmpRet', 'humidity'],
    'label': label_col,
    'label_type': 'anomaly_type',
    'sub_entity': sub_entity_col,
}

ds = TimeSeriesDataset(hvac_df, col_map)
print(ds.anomaly_summary())
my_model = EuclideanDistModel(feature_col='TmpRet')
scores_df = my_model.score_anomalies(ds, level='timestamp')

  label_type  entity_count
0        lag             1
1     normal             1


In [17]:
fig = scores_df.sample_and_visualize_cases()[0]
fig.show()

In [19]:
day_scores = my_model.score_anomalies(ds, level='day')
day_scores.df

,container_id,day,max_eucl_dist,anomaly_score
0,0,2026-01-15,20.824472,0.168093
1,0,2026-01-16,18.777687,-0.121178
2,0,2026-01-17,66.266087,6.590333
3,0,2026-01-18,76.214556,7.996345
4,0,2026-01-19,35.317796,2.216427
5,1,2026-01-15,23.613056,0.562202
6,1,2026-01-16,24.284435,0.657088
7,1,2026-01-17,24.709451,0.717155
8,1,2026-01-18,25.162624,0.781202
9,1,2026-01-19,21.874324,0.316468


In [18]:
ds.df[ds.df['anomaly_type'] == 'lag']

,timestamp_et,HVACNum,TmpRet,anomaly,anomaly_type,container_id,humidity
10081,2026-01-17 08:00:00,1,53.000100,True,lag,0,34.601953
10084,2026-01-17 08:01:00,1,52.792786,True,lag,0,33.817681
10087,2026-01-17 08:02:00,1,52.965496,True,lag,0,31.692105
10090,2026-01-17 08:03:00,1,53.095553,True,lag,0,31.109902
10093,2026-01-17 08:04:00,1,53.052970,True,lag,0,38.886309
...,...,...,...,...,...,...,...
18706,2026-01-19 07:55:00,1,49.546053,True,lag,0,38.990605
18709,2026-01-19 07:56:00,1,49.489377,True,lag,0,37.719113
18712,2026-01-19 07:57:00,1,49.557388,True,lag,0,37.008860
18715,2026-01-19 07:58:00,1,49.709092,True,lag,0,34.458278


In [21]:
fig = ds.sample_and_visualize_cases(entity_ids=[0])[0]
fig.show()

In [33]:
import plotly.express as px
df = scores_df.df
df = df[df['container_id'] == 0]
px.scatter(df, x='timestamp_et', y='anomaly_score')

In [34]:
df['anomaly_score'].mean()

0.542793312813448

In [35]:
df['anomaly_score'].quantile(0.9)

5.745896453806065

In [26]:

px.histogram(df['anomaly_score'])

In [37]:
import plotly.express as px
df = scores_df.df
df = df[df['container_id'] == 1]
fig = px.scatter(df, x='timestamp_et', y='anomaly_score')
fig.show()
print(df['anomaly_score'].quantile(0.9))
px.histogram(df['anomaly_score'])

0.6313555279648363


# Evaluation Class Test
- test with a single euclidean distance model
- test with two different versions of eucl dist model

In [ ]:
# setup of data and model

hvac_df = pd.read_parquet(Path("../datasets/hvac_anomalies_v021226.parquet"))


col_map = {
    'entity': 'container_id',
    'time': 'timestamp_et',
    'value_cols': ['TmpRet'],
    'label': 'anomaly',
    'label_type': 'anomaly_type',
    'sub_entity': 'unit',
}

ds = TimeSeriesDataset(hvac_df, col_map)
print(ds.anomaly_summary())
#TODO add str method for model name

# model with default dist window
my_model_v1 = EuclideanDistModel(feature_col='TmpRet', strategy='iqr')
scores_model_v1 = my_model_v1.score_anomalies_v2(ds)


# V2: with larger dist window
my_model_v2 = EuclideanDistModel(feature_col='TmpRet', strategy='iqr', dist_window = 12*60)
scores_model_v2 = my_model_v2.score_anomalies_v2(ds)

# Eval Test
eval = Evaluation(level="day")
fig = eval.plot_pr_curve(scores_model_v1, ds, model_name="Eucl_Distance_model_v1")
fig.show()


  label_type  entity_count
0     normal           911
1        lag            38
2  amplitude            27
3  frequency            24


TypeError: Evaluation.plot_pr_curves_compared() got an unexpected keyword argument 'scores'

In [10]:

my_model_v3 = EuclideanDistModel(feature_col='TmpRet', strategy='iqr', dist_window = 24*60)
scores_model_v3 = my_model_v3.score_anomalies_v2(ds)

In [11]:

eval.plot_pr_curves_compared(scores_dict={"v1": scores_model_v1, "v2": scores_model_v2, "v3": scores_model_v3}, dataset=ds)

In [23]:
ds.day_labels()

,container_id,day,anomaly,anomaly_type
0,0,2026-01-15,False,normal
1,0,2026-01-16,False,normal
2,0,2026-01-17,True,lag
3,0,2026-01-18,True,lag
4,0,2026-01-19,True,lag
5,1,2026-01-15,False,normal
6,1,2026-01-16,False,normal
7,1,2026-01-17,False,normal
8,1,2026-01-18,False,normal
9,1,2026-01-19,False,normal


In [25]:
scores_model_1.df

,container_id,timestamp_et,max_eucl_dist,anomaly_score
0,0,2026-01-15 01:08:00,10.230273,-1.329180
1,0,2026-01-15 01:09:00,10.237046,-1.328222
2,0,2026-01-15 01:10:00,10.241981,-1.327525
3,0,2026-01-15 01:11:00,10.254580,-1.325744
4,0,2026-01-15 01:12:00,10.262427,-1.324635
...,...,...,...,...
14259,1,2026-01-19 23:55:00,18.577911,-0.149412
14260,1,2026-01-19 23:56:00,18.575583,-0.149741
14261,1,2026-01-19 23:57:00,18.578951,-0.149265
14262,1,2026-01-19 23:58:00,18.578521,-0.149326


In [ ]:
# Test with different anomaly Types: Recreate plot from eval notebook